# DisorderNet v8 — Multi-scale PLM ensemble + calibrated conformal confidence (GPU)

This notebook runs the **updated DisorderNet methods** on a GPU:

1. GPU-accelerated ESM-2 embedding extraction for **multiple backbones** (35M / 150M / 650M).
2. The **v7** optimized CPU model per backbone (leakage-free 5-fold CV, train-only PCA,
   LightGBM+XGBoost+HistGBM blend, contiguity smoothing).
3. The **v8 multi-scale ensemble** of the per-backbone out-of-fold predictions
   (equal weights → leakage-free): the best honest CPU result (~0.857 AUC).
4. **Calibrated probabilities** (isotonic) + **split-conformal** confident/abstain decisions.
5. A **deployable predictor** and a demo prediction on real proteins.

For the GPU **LoRA** SOTA path (targets ≥0.88, now with the same calibrated conformal
confidence auto-attached to its CV summary) use `DisorderNet_Colab_Pro.ipynb`.

**Runtime → Change runtime type → GPU (A100 or L4) + High RAM.**


## 1 · Setup


In [ ]:
# Clone the repo (Colab downloads only the notebook by default) and install CPU deps.
import os, sys, subprocess
REPO_URL = 'https://github.com/Tommaso-R-Marena/DisorderNet.git'
if not os.path.isfile('run_v8_multiscale.py'):
    if not os.path.isdir('DisorderNet'):
        subprocess.run(['git','clone','--depth','1',REPO_URL], check=True)
    os.chdir('DisorderNet')
sys.path.insert(0, '.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-cpu.txt'], check=True)
rev = subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip()
print('DisorderNet @', rev)


In [ ]:
# Confirm GPU is available (embedding extraction uses it automatically).
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — extraction will run on CPU (much slower).')


## 2 · Configuration

Pick which ESM-2 backbones to ensemble. 35M+150M+650M is the paper configuration.
Set `USE_DRIVE=True` to persist embeddings/results across sessions.


In [ ]:
import os
# Full ESM-2 model names -> short label used for the per-backbone home dir.
BACKBONES = {
    'esm2_t12_35M_UR50D':  '35m',
    'esm2_t30_150M_UR50D': '150m',
    'esm2_t33_650M_UR50D': '650m',
}
USE_DRIVE = False           # True => persist under Google Drive
RUN_HOMOLOGY = True         # also run the CAID-credible homology-split CV

if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/DisorderNet/v8'
else:
    BASE = '/content/dn_v8'
os.makedirs(os.path.join(BASE,'data'), exist_ok=True)
DISPROT = os.path.join(BASE,'data','disprot_processed.json')
print('Base dir:', BASE)


## 3 · Fetch DisProt


In [ ]:
import subprocess, sys, os
env = dict(os.environ, DISORDERNET_HOME=BASE)
subprocess.run([sys.executable,'fetch_disprot.py'], env=env, check=True)
print('DisProt cached at', DISPROT, '->', os.path.exists(DISPROT))


## 4 · Extract ESM-2 embeddings for each backbone (GPU)

Uses `extract_embeddings.py` with length-sorted GPU batching. On an A100 each
backbone is minutes; 650M is the slowest. Re-running skips existing `.npy` files.


In [ ]:
import subprocess, sys, os
for model_name, label in BACKBONES.items():
    home = os.path.join(BASE, label)
    emb  = os.path.join(home, 'data', 'embeddings')
    os.makedirs(os.path.join(home,'data'), exist_ok=True)
    # share the single DisProt cache across backbones
    link = os.path.join(home,'data','disprot_processed.json')
    if not os.path.exists(link):
        os.symlink(DISPROT, link)
    print('==> extracting', model_name, '->', emb)
    subprocess.run([sys.executable,'extract_embeddings.py',
                    '--model', model_name, '--data', DISPROT, '--out', emb], check=True)


## 5 · v7 per-backbone CV (random split)


In [ ]:
import subprocess, sys, os
for label in BACKBONES.values():
    home = os.path.join(BASE, label)
    print('==> run_v7 (random split):', label)
    subprocess.run([sys.executable,'run_v7.py'],
                   env=dict(os.environ, DISORDERNET_HOME=home), check=True)


## 6 · v8 multi-scale ensemble (+ calibration + conformal)


In [ ]:
import subprocess, sys, os
backbone_dirs = [os.path.join(BASE, label, 'results_v7') for label in BACKBONES.values()]
subprocess.run([sys.executable,'run_v8_multiscale.py','--backbones',*backbone_dirs],
               env=dict(os.environ, DISORDERNET_HOME=os.path.join(BASE, list(BACKBONES.values())[0]),
                        DISORDERNET_RESULTS_ROOT=os.path.join(BASE,'ensemble')), check=True)


## 7 · (optional) Honest homology-split evaluation


In [ ]:
import subprocess, sys, os
if RUN_HOMOLOGY:
    for label in BACKBONES.values():
        home = os.path.join(BASE, label)
        print('==> run_v7 (homology split):', label)
        subprocess.run([sys.executable,'run_v7.py'],
                       env=dict(os.environ, DISORDERNET_HOME=home, DISORDERNET_SPLIT='homology',
                                DISORDERNET_RESULTS_ROOT=os.path.join(home,'hom')), check=True)
    hom_dirs = [os.path.join(BASE, label,'hom','results_v7') for label in BACKBONES.values()]
    subprocess.run([sys.executable,'run_v8_multiscale.py','--backbones',*hom_dirs],
                   env=dict(os.environ, DISORDERNET_HOME=os.path.join(BASE, list(BACKBONES.values())[0]),
                            DISORDERNET_RESULTS_ROOT=os.path.join(BASE,'ensemble_hom')), check=True)


## 8 · Reliability + conformal figure (ensemble)


In [ ]:
import json, numpy as np, os
import matplotlib.pyplot as plt
from confidence import expected_calibration_error
R = os.path.join(BASE,'ensemble','results_v8')
yt = np.load(os.path.join(R,'y_true.npy')); yp = np.load(os.path.join(R,'y_pred.npy'))
yc = np.load(os.path.join(R,'y_pred_calibrated.npy'))
M = json.load(open(os.path.join(R,'metrics.json')))
print('ensemble AUC:', round(M['ensemble']['auc_roc'],4))
print('ECE:', round(M['calibration']['ece_before'],4),'->',round(M['calibration']['ece_after'],4))
print('conformal coverage:', round(M['conformal']['coverage'],3))
def rel(y,p,nb=15):
    b=np.linspace(0,1,nb+1); idx=np.clip(np.digitize(p,b[1:-1]),0,nb-1)
    xs=[];ys=[]
    for k in range(nb):
        m=idx==k
        if m.sum()>30: xs.append(p[m].mean()); ys.append(y[m].mean())
    return xs,ys
plt.figure(figsize=(6,6)); plt.plot([0,1],[0,1],'k--',alpha=.4)
x0,y0=rel(yt,yp); x1,y1=rel(yt,yc)
plt.plot(x0,y0,'o-',label='uncalibrated'); plt.plot(x1,y1,'o-',label='isotonic')
plt.xlabel('predicted p(disorder)'); plt.ylabel('observed'); plt.legend(); plt.title('Reliability (v8 ensemble)')
plt.show()


## 9 · Deployable predictor + demo prediction


In [ ]:
import subprocess, sys, os
home35 = os.path.join(BASE, '35m')  # bundle uses ESM-2 35M embeddings (fast at inference)
subprocess.run([sys.executable,'train_predictor.py'],
               env=dict(os.environ, DISORDERNET_HOME=home35), check=True)
asyn = 'MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEGAGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA'
subprocess.run([sys.executable,'predict_disorder.py','--seq',asyn],
               env=dict(os.environ, DISORDERNET_HOME=home35), check=True)


## GPU LoRA path (≥0.88 target)

For the SOTA fine-tuned models, open **`DisorderNet_Colab_Pro.ipynb`** and run the
ultra/ultra3b profiles. `run_cross_validation` now automatically attaches a
cross-fitted **calibration + conformal** report to its CV summary (ECE before/after,
coverage, selective accuracy), so the GPU models get the same trustworthy per-residue
uncertainty as the CPU ensemble.
